<a href="https://colab.research.google.com/github/idarapatrick/Dual-Encoder-Model-for-Sahel-Dust-Forecasting/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SahelDust Preprocessing Pipeline

This notebook extracts 72-hour ERA5 hourly sequences, same-day SMAP soil moisture, and MODIS AOD labels from Google Earth Engine, then normalizes and saves the aligned dataset.

## Data sources
- ERA5 hourly (ECMWF/ERA5/HOURLY): u10, v10, t2m, surface pressure
- SMAP soil moisture AM (NASA/SMAP/SPL3SMP_E/005 and 006)
- MODIS AOD (MODIS/061/MCD19A2_GRANULES): Optical_Depth_055
- AERONET proxy: MODIS AOD at 5 ground station locations

## Temporal window
- T-60 to T+12 hours relative to event day midnight
- SMAP uses same-day AM retrieval (06:00 local, before emission window)

## Temporal split
- Training: 2015-2022 (batches 001-086)
- Validation: 2023 (batches 087-097)
- Test: 2024 full unsubsampled (saheldust_test_full folder)

In [4]:
!earthengine authenticate

Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=aiMJzBsvHHCrVRPIzKfHg7TnuVnvXTralcRQtS3xwN4&tc=FOtwhqSHpfppRUcZm3fFKyI204rItXCLGfMLq4JG6EE&cc=jTS-4KntfSFl5ACG2Y2j1IbszWcZQe6wVyFXHrt82AM

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPwZcH1TslKc61ouf8ISyhz_dVypbRUWAwjChPuTT0Hcz-MYQJCjfP4

Successfully saved authorization token.


In [5]:
import ee

try:
    ee.Initialize(project='sahel-dust-forecasting')
    print("GEE authenticated successfully")
    print(ee.String('Connection confirmed').getInfo())
except Exception as e:
    print(f"Authentication needed: {e}")

GEE authenticated successfully
Connection confirmed


Installation and Authentication

In [6]:
!pip install earthengine-api -q

import ee
import numpy as np
import os
import json
import time
import glob
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

ee.Initialize(project='sahel-dust-forecasting')

from google.colab import drive
drive.mount('/content/drive')

print("completed")

Mounted at /content/drive
completed


Constants and grid

In [7]:
# Sahel bounding box
SAHEL = ee.Geometry.Rectangle([-18, 10, 25, 25])

# Grid dimensions
TARGET_H, TARGET_W = 68, 172
N_PIXELS = TARGET_H * TARGET_W  # 11,696

# computePixels grid specification
# translateX/Y define the top-left corner of the grid
# scaleY is negative because rows go north to south
GRID = {
    'dimensions': {'width': TARGET_W, 'height': TARGET_H},
    'affineTransform': {
        'scaleX':  0.25, 'shearX': 0, 'translateX': -18.125,
        'shearY':  0,    'scaleY': -0.25, 'translateY': 25.125
    },
    'crsCode': 'EPSG:4326'
}

# Coordinate grids
lons_1d  = np.linspace(-18, 25, TARGET_W)
lats_1d  = np.linspace(25,  10, TARGET_H)
LON_GRID, LAT_GRID = np.meshgrid(lons_1d, lats_1d)

# Paths
BASE_DIR  = '/content/drive/MyDrive/saheldust_data'
OUT_DIR   = f'{BASE_DIR}/saheldust_processed_hourly'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_FILE = f'{OUT_DIR}/checkpoint.json'

# Subsampling
# Keep ALL dust event pixels.
# Keep 5% of non-event pixels to keep dataset manageable.
# Decision: 5% gives roughly balanced training after weighting.
NON_EVENT_RATE = 0.05

# Parallelisation
# 4 concurrent workers = 4x speedup.
# Reduce to 2 if you see GEE rate limit errors.
N_WORKERS = 4

# Batch checkpoint frequency
SAVE_EVERY_N_DAYS = 30

print(f"Grid:          {TARGET_H} x {TARGET_W} = {N_PIXELS:,} pixels")
print(f"Output dir:    {OUT_DIR}")
print(f"Non-event rate: {NON_EVENT_RATE*100:.0f}%")
print(f"Workers:       {N_WORKERS}")

Grid:          68 x 172 = 11,696 pixels
Output dir:    /content/drive/MyDrive/saheldust_data/saheldust_processed_hourly
Non-event rate: 5%
Workers:       4


Functions to extract GEE

In [8]:
def fetch_pixels(ee_image):
    """
    Extract a GEE image as a numpy structured array using computePixels.
    Returns None on failure.
    """
    try:
        return ee.data.computePixels({
            'expression': ee_image,
            'fileFormat': 'NUMPY_NDARRAY',
            'grid': GRID
        })
    except Exception as e:
        return None


def get_era5_window(day):
    """
    Fetch the 72-hour ERA5 window for one day.

    Window: T-60 to T+12 where T = midnight UTC of event day.
    This captures:
      - 2.5 days of atmospheric build-up before the event
      - The first 12 hours of the event day (pre-emission period)

    Returns numpy array of shape (H, W, 72, 4) or None.
    Variable order in axis 3: [u10, v10, t2m, sp]
    """
    t0    = ee.Date(day.strftime('%Y-%m-%dT00:00:00'))
    t_start = t0.advance(-60, 'hour')
    t_end   = t0.advance( 12, 'hour')

    era5 = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
              .filterDate(t_start, t_end)
              .select([
                  'u_component_of_wind_10m',
                  'v_component_of_wind_10m',
                  'temperature_2m',
                  'surface_pressure'
              ]))

    # Verify we have all 72 hours
    n = era5.size().getInfo()
    if n < 72:
        return None

    # Clip to exactly 72 in chronological order
    era5 = ee.ImageCollection(era5.sort('system:time_start').toList(72))

    # Stack each variable separately into 72-band images
    # then combine all into one 288-band image for a single API call
    u10 = era5.select('u_component_of_wind_10m').toBands()
    v10 = era5.select('v_component_of_wind_10m').toBands()
    t2m = era5.select('temperature_2m').toBands()
    sp  = era5.select('surface_pressure').toBands()

    combined = (u10.rename([f'u_{i}' for i in range(72)])
                   .addBands(v10.rename([f'v_{i}' for i in range(72)]))
                   .addBands(t2m.rename([f't_{i}' for i in range(72)]))
                   .addBands(sp.rename( [f's_{i}' for i in range(72)])))

    pixels = fetch_pixels(combined)
    if pixels is None:
        return None

    # Unpack structured array into (4, 72, H, W)
    u_arr = np.stack([pixels[f'u_{i}'] for i in range(72)], axis=0)
    v_arr = np.stack([pixels[f'v_{i}'] for i in range(72)], axis=0)
    t_arr = np.stack([pixels[f't_{i}'] for i in range(72)], axis=0)
    s_arr = np.stack([pixels[f's_{i}'] for i in range(72)], axis=0)

    # Shape: (72, H, W, 4) — timesteps first, variables last
    era5_array = np.stack([u_arr, v_arr, t_arr, s_arr],
                           axis=-1).astype(np.float32)
    # era5_array[t, row, col, var] — (72, H, W, 4)
    # Transpose to (H, W, 72, 4) for pixel-level indexing
    return era5_array.transpose(1, 2, 0, 3)


def get_smap_day(day):
    """
    Fetch same-day SMAP AM soil moisture.

    Decision: using same-day SMAP because the AM overpass
    (~06:00 local time) precedes the primary Sahel dust emission
    window (09:00-14:00 local time). This is physically prior to
    the event and is not data leakage.

    Collections switch at 2023-12-04 (confirmed from GEE catalog).
    Returns numpy array of shape (H, W) or None.
    """
    coll_id = ('NASA/SMAP/SPL3SMP_E/005'
               if day < datetime(2023, 12, 4)
               else 'NASA/SMAP/SPL3SMP_E/006')

    day_start = ee.Date(day.strftime('%Y-%m-%d'))
    day_end   = day_start.advance(1, 'day')

    smap = (ee.ImageCollection(coll_id)
              .filterDate(day_start, day_end)
              .select('soil_moisture_am'))

    if smap.size().getInfo() == 0:
        return None

    pixels = fetch_pixels(smap.first())
    if pixels is None:
        return None

    arr = pixels['soil_moisture_am'].astype(np.float32)
    arr[arr < 0] = np.nan   # negative = fill value
    return arr


def get_modis_label(day):
    """
    Fetch MODIS AOD and derive binary dust event label.

    Threshold: AOD > 0.5 at 550 nm (as specified in thesis).
    Scale factor: raw × 0.001 = actual AOD.
    Uses daily maximum across granules to capture peak dust loading.

    Returns numpy array of shape (H, W) with values 0.0/1.0 or None.
    """
    day_start = ee.Date(day.strftime('%Y-%m-%d'))
    day_end   = day_start.advance(1, 'day')

    modis = (ee.ImageCollection('MODIS/061/MCD19A2_GRANULES')
               .filterDate(day_start, day_end)
               .select('Optical_Depth_055'))

    if modis.size().getInfo() == 0:
        return None

    aod_raw_img = modis.max()
    pixels = fetch_pixels(aod_raw_img)
    if pixels is None:
        return None

    raw = pixels['Optical_Depth_055'].astype(np.float32)
    aod = raw * 0.001
    label = (aod > 0.5).astype(np.float32)
    label[raw <= 0] = np.nan   # no valid retrieval
    return label

Function to process data per day

In [9]:
def process_day(day):
    """
    Process one day end-to-end.
    Returns a list of (atm, surf, label) tuples for valid pixels.
    Returns empty list if any dataset is unavailable for that day.

    This function is called by the parallel workers.
    """
    try:
        label_grid = get_modis_label(day)
        if label_grid is None:
            return day, []

        smap_grid = get_smap_day(day)
        if smap_grid is None:
            return day, []

        era5_grid = get_era5_window(day)   # (H, W, 72, 4)
        if era5_grid is None:
            return day, []

        month     = day.month
        month_sin = np.float32(np.sin(2 * np.pi * month / 12))
        month_cos = np.float32(np.cos(2 * np.pi * month / 12))
        day_str   = day.strftime('%Y-%m-%d')

        samples = []
        rng = np.random.default_rng(seed=int(day.strftime('%Y%m%d')))

        for row in range(TARGET_H):
            for col in range(TARGET_W):
                atm = era5_grid[row, col]         # (72, 4)
                sm  = smap_grid[row, col]         # scalar
                lbl = label_grid[row, col]        # scalar

                # Skip pixels with any missing value
                if (np.isnan(sm) or np.isnan(lbl)
                        or np.any(np.isnan(atm))):
                    continue

                # Keep all dust events.
                # Subsample non-events at NON_EVENT_RATE.
                if lbl == 0.0 and rng.random() > NON_EVENT_RATE:
                    continue

                lat = np.float32(LAT_GRID[row, col])
                lon = np.float32(LON_GRID[row, col])
                surf = np.array(
                    [sm, lat, lon, month_sin, month_cos],
                    dtype=np.float32
                )
                samples.append((atm.copy(), surf, np.float32(lbl)))

        return day, samples

    except Exception as e:
        return day, []

Checkpoint functions

In [10]:
def load_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'r') as f:
            return json.load(f)
    return {'last_date': None, 'total_samples': 0, 'next_batch': 1}


def save_checkpoint(last_date, total_samples, next_batch):
    with open(CKPT_FILE, 'w') as f:
        json.dump({
            'last_date':     last_date.strftime('%Y-%m-%d'),
            'total_samples': total_samples,
            'next_batch':    next_batch
        }, f)
    print(f"  Checkpoint: {last_date.strftime('%Y-%m-%d')} | "
          f"samples={total_samples:,} | next_batch={next_batch}")


def flush_batch(atm_buf, surf_buf, label_buf, batch_idx):
    """Save current buffer to Drive and return empty buffers."""
    if not atm_buf:
        return [], [], []
    path = f'{OUT_DIR}/batch_{batch_idx:04d}.npz'
    np.savez_compressed(
        path,
        X_atm  = np.array(atm_buf,   dtype=np.float32),
        X_surf = np.array(surf_buf,  dtype=np.float32),
        y      = np.array(label_buf, dtype=np.float32)
    )
    print(f"  Batch {batch_idx:04d} saved: {len(atm_buf):,} samples → {path}")
    return [], [], []

Split using year tracking

In [11]:
# The batch files as written do not store per-sample dates.
# since each batch covers SAVE_EVERY_N_DAYS days
# and samples are in chronological order within each batch.
# For the temporal split, only the year is needed.
# Add year tracking to the flush step.

# flush function
def flush_batch_with_years(atm_buf, surf_buf, label_buf,
                           year_buf, batch_idx):
    if not atm_buf:
        return [], [], [], []
    path = f'{OUT_DIR}/batch_{batch_idx:04d}.npz'
    np.savez_compressed(
        path,
        X_atm  = np.array(atm_buf,   dtype=np.float32),
        X_surf = np.array(surf_buf,  dtype=np.float32),
        y      = np.array(label_buf, dtype=np.float32),
        years  = np.array(year_buf,  dtype=np.int16)
    )
    size_mb = os.path.getsize(path) / 1e6
    print(f"  Batch {batch_idx:04d}: {len(atm_buf):,} samples "
          f"({size_mb:.1f} MB)")
    return [], [], [], []

Main parallel processing loop

In [10]:
np.random.seed(42)

# Build date list
# Start April 2015 (first complete SMAP month)
all_days = []
d = datetime(2015, 4, 1)
while d <= datetime(2024, 12, 31):
    all_days.append(d)
    d += timedelta(days=1)

print(f"Total days to process: {len(all_days):,}")

# Resume from checkpoint
ckpt = load_checkpoint()
if ckpt['last_date']:
    resume_from = datetime.strptime(ckpt['last_date'], '%Y-%m-%d')
    all_days    = [d for d in all_days if d > resume_from]
    print(f"Resuming from: {ckpt['last_date']}")
    print(f"Remaining days: {len(all_days):,}")

total_samples = ckpt['total_samples']
next_batch    = ckpt['next_batch']

atm_buf, surf_buf, label_buf = [], [], []
days_since_flush = 0
n_skipped = 0
t_start_wall = time.time()

# Process in parallel batches of N_WORKERS days
# submit N_WORKERS days at a time, wait for all to finish,
# then submit the next N_WORKERS days.
# This controls memory and avoids overwhelming GEE.

day_batches = [all_days[i:i+N_WORKERS]
               for i in range(0, len(all_days), N_WORKERS)]

for batch_idx_loop, day_batch in enumerate(day_batches):

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_day, day): day
                   for day in day_batch}

        for future in as_completed(futures):
            day, samples = future.result()

            if not samples:
                n_skipped += 1
                continue

            for atm, surf, lbl in samples:
                atm_buf.append(atm)
                surf_buf.append(surf)
                label_buf.append(lbl)

            total_samples += len(samples)
            days_since_flush += 1

    # Progress report every batch
    elapsed = (time.time() - t_start_wall) / 3600
    days_done = (batch_idx_loop + 1) * N_WORKERS
    rate = days_done / max(elapsed, 0.01)
    remaining_days = len(all_days) - days_done
    eta_hours = remaining_days / max(rate, 1) / 3600

    print(f"[{day_batch[-1].strftime('%Y-%m-%d')}] "
          f"days={days_done:,} | "
          f"samples={total_samples:,} | "
          f"buf={len(atm_buf):,} | "
          f"elapsed={elapsed:.1f}h | "
          f"ETA={eta_hours:.1f}h")

    # Flush to Drive every SAVE_EVERY_N_DAYS
    if days_since_flush >= SAVE_EVERY_N_DAYS:
        atm_buf, surf_buf, label_buf = flush_batch(
            atm_buf, surf_buf, label_buf, next_batch
        )
        save_checkpoint(day_batch[-1], total_samples, next_batch + 1)
        next_batch       += 1
        days_since_flush  = 0

# Flush final partial batch
if atm_buf:
    flush_batch(atm_buf, surf_buf, label_buf, next_batch)
    save_checkpoint(datetime(2024, 12, 31), total_samples, next_batch + 1)

print(f"\nDone. Total samples: {total_samples:,} | Skipped: {n_skipped:,}")

Total days to process: 3,563
Resuming from: 2015-07-05
Remaining days: 3,467
[2015-07-09] days=4 | samples=66,536 | buf=1,433 | elapsed=0.0h | ETA=0.0h
[2015-07-13] days=8 | samples=68,485 | buf=3,382 | elapsed=0.0h | ETA=0.0h
[2015-07-17] days=12 | samples=70,290 | buf=5,187 | elapsed=0.0h | ETA=0.0h
[2015-07-21] days=16 | samples=72,832 | buf=7,729 | elapsed=0.0h | ETA=0.0h
[2015-07-25] days=20 | samples=74,595 | buf=9,492 | elapsed=0.0h | ETA=0.0h
[2015-07-29] days=24 | samples=75,981 | buf=10,878 | elapsed=0.0h | ETA=0.0h
[2015-08-02] days=28 | samples=78,207 | buf=13,104 | elapsed=0.0h | ETA=0.0h
[2015-08-06] days=32 | samples=80,203 | buf=15,100 | elapsed=0.0h | ETA=0.0h
  Batch 0004 saved: 15,100 samples → /content/drive/MyDrive/saheldust_data/saheldust_processed_hourly/batch_0004.npz
  Checkpoint: 2015-08-06 | samples=80,203 | next_batch=5
[2015-08-10] days=36 | samples=83,184 | buf=2,981 | elapsed=0.0h | ETA=0.0h
[2015-08-14] days=40 | samples=85,937 | buf=5,734 | elapsed=0.0h

Fix the GEE pipeline for 2024 test set, keeping all pixels regardless of whether they are dust events or non events.

In [16]:
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

ee.Initialize(project='sahel-dust-forecasting')

# New output folder for full test set
TEST_DIR = f'{BASE_DIR}/saheldust_test_full'
os.makedirs(TEST_DIR, exist_ok=True)

# Override subsampling: keep ALL pixels
# This gives a realistic dust event distribution for evaluation
NON_EVENT_RATE_TEST = 1.0

def process_day_full(day):
    """
    Same as process_day but keeps all valid pixels.
    No subsampling. Used for the test set only.
    """
    try:
        label_grid = get_modis_label(day)
        if label_grid is None:
            return day, []

        smap_grid = get_smap_day(day)
        if smap_grid is None:
            return day, []

        era5_grid = get_era5_window(day)
        if era5_grid is None:
            return day, []

        month     = day.month
        month_sin = np.float32(np.sin(2 * np.pi * month / 12))
        month_cos = np.float32(np.cos(2 * np.pi * month / 12))

        samples = []
        for row in range(TARGET_H):
            for col in range(TARGET_W):
                atm = era5_grid[row, col]
                sm  = smap_grid[row, col]
                lbl = label_grid[row, col]
                lat = np.float32(LAT_GRID[row, col])
                lon = np.float32(LON_GRID[row, col])

                # Skip only genuinely missing values
                if (np.isnan(sm) or np.isnan(lbl)
                        or np.any(np.isnan(atm))):
                    continue

                # Keep ALL valid pixels and no subsampling
                surf = np.array(
                    [sm, lat, lon, month_sin, month_cos],
                    dtype=np.float32
                )
                samples.append((atm.copy(), surf, np.float32(lbl)))

        return day, samples

    except Exception as e:
        return day, []


# Generate 2024 date list
test_days = []
d = datetime(2024, 1, 1)
while d <= datetime(2024, 12, 31):
    test_days.append(d)
    d += timedelta(days=1)

print(f"2024 test days to process: {len(test_days)}")

# Process in parallel
test_atm_buf, test_surf_buf, test_y_buf = [], [], []
n_samples_test = 0
batch_idx_test = 1
t_start = time.time()

day_batches = [test_days[i:i+N_WORKERS]
               for i in range(0, len(test_days), N_WORKERS)]

for batch_i, day_batch in enumerate(day_batches):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_day_full, day): day
                   for day in day_batch}
        for future in as_completed(futures):
            day, samples = future.result()
            if not samples:
                continue
            for atm, surf, lbl in samples:
                test_atm_buf.append(atm)
                test_surf_buf.append(surf)
                test_y_buf.append(lbl)
            n_samples_test += len(samples)

    elapsed = (time.time() - t_start) / 3600
    print(f"[{day_batch[-1].strftime('%Y-%m-%d')}] "
          f"days={( batch_i+1)*N_WORKERS} | "
          f"samples={n_samples_test:,} | "
          f"elapsed={elapsed:.2f}h")

    # Save every 30 days
    if (batch_i + 1) % (30 // N_WORKERS) == 0 and test_atm_buf:
        path = f'{TEST_DIR}/test_batch_{batch_idx_test:03d}.npz'
        np.savez_compressed(
            path,
            X_atm  = np.array(test_atm_buf,  dtype=np.float32),
            X_surf = np.array(test_surf_buf, dtype=np.float32),
            y      = np.array(test_y_buf,    dtype=np.float32)
        )
        print(f"  Saved {path} | {len(test_y_buf):,} samples")
        test_atm_buf, test_surf_buf, test_y_buf = [], [], []
        batch_idx_test += 1

# Save final batch
if test_atm_buf:
    path = f'{TEST_DIR}/test_batch_{batch_idx_test:03d}.npz'
    np.savez_compressed(
        path,
        X_atm  = np.array(test_atm_buf,  dtype=np.float32),
        X_surf = np.array(test_surf_buf, dtype=np.float32),
        y      = np.array(test_y_buf,    dtype=np.float32)
    )
    print(f"  Saved {path} | {len(test_y_buf):,} samples")

print(f"\n2024 full test set complete: {n_samples_test:,} total samples")

2024 test days to process: 366
[2024-01-04] days=4 | samples=16,781 | elapsed=0.00h
[2024-01-08] days=8 | samples=31,274 | elapsed=0.00h
[2024-01-12] days=12 | samples=47,610 | elapsed=0.00h
[2024-01-16] days=16 | samples=60,826 | elapsed=0.01h
[2024-01-20] days=20 | samples=78,315 | elapsed=0.01h
[2024-01-24] days=24 | samples=92,865 | elapsed=0.01h
[2024-01-28] days=28 | samples=110,066 | elapsed=0.01h
  Saved /content/drive/MyDrive/saheldust_data/saheldust_test_full/test_batch_001.npz | 110,066 samples
[2024-02-01] days=32 | samples=126,168 | elapsed=0.01h
[2024-02-05] days=36 | samples=141,816 | elapsed=0.01h
[2024-02-09] days=40 | samples=157,039 | elapsed=0.01h
[2024-02-13] days=44 | samples=174,783 | elapsed=0.01h
[2024-02-17] days=48 | samples=191,227 | elapsed=0.01h
[2024-02-21] days=52 | samples=208,257 | elapsed=0.02h
[2024-02-25] days=56 | samples=221,263 | elapsed=0.02h
  Saved /content/drive/MyDrive/saheldust_data/saheldust_test_full/test_batch_002.npz | 111,197 samples
[

In [47]:
!kaggle datasets list --mine

ref                                title                        size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------  ---------------------  ----------  --------------------------  -------------  ---------  ---------------  
avalarcodes/beijing-air-quality    beijing-air-quality        859445  2026-01-24 08:37:13.600000              0          0  0.11764706       
avalarcodes/uci-drug-review        uci-drug-review          40484738  2026-02-07 22:28:54.040000              0          0  0.11764706       
avalarcodes/drug-review-src-utils  drug-review-src-utils        5944  2026-02-07 22:51:55.133000              0          0  0.0              
avalarcodes/medical-checkpoint     medical-checkpoint       62895179  2026-02-22 17:52:02.080000              0          0  0.25             


In [48]:
import json

OUT_DIR = '/content/drive/MyDrive/saheldust_data/saheldust_processed_hourly'

!mkdir -p /content/saheldust_final_upload
!cp "{OUT_DIR}/sahel_hourly_preprocessed.npz" /content/saheldust_final_upload/

meta = {
    "title": "SahelDust Hourly Preprocessed Dataset",
    "id": "avalarcodes/saheldust-hourly-preprocessed",
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/content/saheldust_final_upload/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

!kaggle datasets create -p /content/saheldust_final_upload --dir-mode zip

Starting upload for file sahel_hourly_preprocessed.npz
100% 1.73G/1.73G [00:16<00:00, 115MB/s]
Upload successful: sahel_hourly_preprocessed.npz (2GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/avalarcodes/saheldust-hourly-preprocessed


--------

Reinstall and Remount

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import glob
import os
import joblib
from sklearn.preprocessing import StandardScaler

BASE = '/content/drive/MyDrive/saheldust_data'
OUT_DIR = f'{BASE}/saheldust_processed_hourly'
TEST_DIR = f'{BASE}/saheldust_test_full'

print("Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready


Load train and validation batches

In [4]:
TRAIN_LAST = 86
VAL_LAST = 97

train_atm, train_surf, train_y = [], [], []
val_atm, val_surf, val_y = [], [], []

batch_files = sorted(glob.glob(f'{OUT_DIR}/batch_*.npz'))
print(f"Found {len(batch_files)} batch files")

for bf in batch_files:
    batch_num = int(os.path.basename(bf).replace('batch_', '').replace('.npz', ''))
    data = np.load(bf)
    X_a = data['X_atm'].astype(np.float32)
    X_s = data['X_surf'].astype(np.float32)
    y_b = data['y'].astype(np.float32)
    data.close()

    if batch_num <= TRAIN_LAST:
        train_atm.append(X_a)
        train_surf.append(X_s)
        train_y.append(y_b)
    elif batch_num <= VAL_LAST:
        val_atm.append(X_a)
        val_surf.append(X_s)
        val_y.append(y_b)

X_atm_tr  = np.concatenate(train_atm);  del train_atm
X_surf_tr = np.concatenate(train_surf); del train_surf
y_tr      = np.concatenate(train_y);    del train_y

X_atm_val  = np.concatenate(val_atm);  del val_atm
X_surf_val = np.concatenate(val_surf); del val_surf
y_val      = np.concatenate(val_y);    del val_y

print(f"Train: {len(y_tr):,} samples | dust rate: {y_tr.mean()*100:.1f}%")
print(f"Val:   {len(y_val):,} samples | dust rate: {y_val.mean()*100:.1f}%")

Found 109 batch files
Train: 1,394,076 samples | dust rate: 67.9%
Val:   158,426 samples | dust rate: 64.2%


Load full unsubsampled test set

In [5]:
test_atm, test_surf, test_y = [], [], []

test_files = sorted(glob.glob(f'{TEST_DIR}/test_batch_*.npz'))
print(f"Found {len(test_files)} test batch files")

for tf in test_files:
    data = np.load(tf)
    test_atm.append(data['X_atm'].astype(np.float32))
    test_surf.append(data['X_surf'].astype(np.float32))
    test_y.append(data['y'].astype(np.float32))
    data.close()

X_atm_te  = np.concatenate(test_atm);  del test_atm
X_surf_te = np.concatenate(test_surf); del test_surf
y_te      = np.concatenate(test_y);    del test_y

print(f"Test: {len(y_te):,} samples | dust rate: {y_te.mean()*100:.1f}%")

Found 14 test batch files
Test: 1,253,328 samples | dust rate: 10.7%


Normalize with training statistics

In [6]:
surf_scaler = StandardScaler()
X_surf_tr_n  = surf_scaler.fit_transform(X_surf_tr)
X_surf_val_n = surf_scaler.transform(X_surf_val)
X_surf_te_n  = surf_scaler.transform(X_surf_te)

atm_scaler = StandardScaler()
N_tr  = len(y_tr)
N_val = len(y_val)
N_te  = len(y_te)

X_atm_tr_n  = atm_scaler.fit_transform(
    X_atm_tr.reshape(-1, 4)).reshape(N_tr, 72, 4).astype(np.float32)
del X_atm_tr

X_atm_val_n = atm_scaler.transform(
    X_atm_val.reshape(-1, 4)).reshape(N_val, 72, 4).astype(np.float32)
del X_atm_val

X_atm_te_n  = atm_scaler.transform(
    X_atm_te.reshape(-1, 4)).reshape(N_te, 72, 4).astype(np.float32)
del X_atm_te

del X_surf_tr, X_surf_val, X_surf_te

print("Normalisation complete")

Normalisation complete


Save corrected file and verify size

In [9]:
out_path = f'{OUT_DIR}/sahel_hourly_preprocessed.npz'

np.savez_compressed(
    out_path,
    X_atm_train=X_atm_tr_n,   X_surf_train=X_surf_tr_n,  y_train=y_tr,
    X_atm_val=X_atm_val_n,    X_surf_val=X_surf_val_n,   y_val=y_val,
    X_atm_test=X_atm_te_n,    X_surf_test=X_surf_te_n,   y_test=y_te
)

joblib.dump(surf_scaler, f'{OUT_DIR}/surf_scaler.pkl')
joblib.dump(atm_scaler,  f'{OUT_DIR}/atm_scaler.pkl')

size_gb = os.path.getsize(out_path) / 1e9
print(f"Saved: {out_path}")
print(f"Size: {size_gb:.2f} GB")
print(f"Train: {len(y_tr):,} | Val: {len(y_val):,} | Test: {len(y_te):,}")
print(f"Test dust rate: {y_te.mean()*100:.1f}%")

Saved: /content/drive/MyDrive/saheldust_data/saheldust_processed_hourly/sahel_hourly_preprocessed.npz
Size: 2.99 GB
Train: 1,394,076 | Val: 158,426 | Test: 1,253,328
Test dust rate: 10.7%


In [10]:
LOCAL_PATH = '/content/sahel_hourly_preprocessed.npz'

np.savez_compressed(
    LOCAL_PATH,
    X_atm_train=X_atm_tr_n,  X_surf_train=X_surf_tr_n,  y_train=y_tr,
    X_atm_val=X_atm_val_n,   X_surf_val=X_surf_val_n,   y_val=y_val,
    X_atm_test=X_atm_te_n,   X_surf_test=X_surf_te_n,   y_test=y_te
)

import os
size = os.path.getsize(LOCAL_PATH) / 1e9
print(f"Local file size: {size:.2f} GB")

Local file size: 2.99 GB
